In [0]:
# -------------------------------------------------------------------------
# SCRIPT: 3_taxa_presenca_calc
# LOCAL: 3_gold/src/2_calendario_eventos/
# OBJETIVO: Calcular a taxa de presença por deputado e tipo de evento
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count, round

# Carregar tabelas
df_presencas = spark.table("silver_presenca_eventos")
df_fato = spark.table("gold_fato_eventos")

# Cruzar por ID do evento para garantir os eventos em questão
df_unificado = df_presencas.alias("p").join(
    df_fato.alias("f"),
    col("p.id_evento") == col("f.id_evento"),
    how="inner"
).select(
    col("p.id").alias("id_deputado"),
    col("p.nome").alias("nome_deputado"),
    col("f.descricao").alias("tipo_evento"),
    col("f.id_evento")
)

# Contando as presenças por deputado e tipo
df_contagem_deputado = df_unificado.groupBy("id_deputado", "nome_deputado", "tipo_evento") \
    .agg(count("id_evento").alias("presencas_confirmadas"))

# Total de eventos que existiram para cada tipo na Fato
df_total_eventos_tipo = df_fato.groupBy("descricao") \
    .agg(count("id_evento").alias("total_eventos_tipo")) \
    .withColumnRenamed("descricao", "tipo_evento")

# Join Final para o cálculo da taxa
df_taxa_final = df_contagem_deputado.join(
    df_total_eventos_tipo, 
    on="tipo_evento", 
    how="inner"
)

# Cálculo da porcentagem
df_taxa_final = df_taxa_final.withColumn(
    "taxa_presenca_perc", 
    round((col("presencas_confirmadas") / col("total_eventos_tipo")) * 100, 2)
)

# Salvar na Gold
df_taxa_final.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_taxa_presenca_deputado")

display(df_taxa_final.orderBy(col("taxa_presenca_perc").desc()))

In [0]:
display(spark.table("gold_fato_eventos").groupBy("descricao").count().orderBy(col("count").desc()))